# Historical Long/Short AAC - Single Environment

Train the non-batched `LongShortEnv`, inspect one full training episode, and then run a deterministic validation replay with portfolio diagnostics.

In [ ]:
import os
import sys
from datetime import datetime

import numpy as np
import torch

sys.path.append("..")

from src.data import load_and_align_data, get_field

from bokeh.palettes import Category10
import bokeh.plotting as bk

bk.output_notebook()

In [ ]:
PAIRS_ = {
    "Bitcoin": "XBTEUR",
    "Ethereum": "ETHEUR",
    "Ripple": "XRPEUR",
    "Cardano": "ADAEUR",
    "Solana": "SOLEUR",
}

PAIRS_

In [ ]:
if os.path.exists("../data/historical_data.ptt"):
    print("reading data from file...")
    _data = torch.load("../data/historical_data.ptt")

    times_ = _data["times"]
    dt = float(times_.diff().mean().round())
    print(f"dt = {dt}")

    close = _data["close"]
    high = _data.get("high", close)
    low = _data.get("low", close)
    open_ = _data.get("open", close)
    volume = _data["volume"]
    PAIRS_ = _data["pairs"]

elif os.path.exists("../data/Kraken_OHLCVT"):
    print("loading and aligning data from raw files...")
    _data, times = load_and_align_data(PAIRS_, interval=5)

    times_ = torch.tensor([t.timestamp() for t in times], dtype=torch.float64)
    dt = float(times_.diff().mean().round())
    print(f"dt = {dt}")

    close = torch.tensor(get_field(_data, "close")).T
    high = torch.tensor(get_field(_data, "high")).T
    low = torch.tensor(get_field(_data, "low")).T
    open_ = torch.tensor(get_field(_data, "open")).T
    volume = torch.tensor(get_field(_data, "volume")).T

else:
    raise FileNotFoundError("No historical data found. Please download and prepare the data as described in the README.")

In [ ]:
close_t = close.to(dtype=torch.float32)
high_t = high.to(dtype=torch.float32)
low_t = low.to(dtype=torch.float32)
open_t = open_.to(dtype=torch.float32)
volume_t = volume.to(dtype=torch.float32)
times_t = times_.to(dtype=torch.float64)

sort_idx = torch.argsort(times_t)
times_t = times_t[sort_idx]
close_t = close_t[sort_idx]
high_t = high_t[sort_idx]
low_t = low_t[sort_idx]
open_t = open_t[sort_idx]
volume_t = volume_t[sort_idx]

T_total = close_t.shape[0]
N_assets = close_t.shape[1]
asset_names = list(PAIRS_.keys()) if isinstance(PAIRS_, dict) else list(PAIRS_)

history = [
    {
        "time": float(times_t[i]),
        "close": close_t[i],
        "high": high_t[i],
        "low": low_t[i],
        "open": open_t[i],
        "volume": volume_t[i],
    }
    for i in range(T_total)
]

print(f"T={T_total}, N={N_assets}, train split={int(T_total * 0.9)}")
asset_names

In [ ]:
from src.environment.discrete.longshort_buckets import LongShortEnv

base_t = 60
tau_p = torch.tensor([base_t * 30, base_t * 60 * 4, base_t * 60 * 24, base_t * 60 * 24 * 7], dtype=torch.float32)
print("tau_p:", (tau_p / (3600 * 24)).tolist(), "[days]")

shared_kwargs = dict(
    N=len(asset_names),
    C0=1_000,
    tau_p=tau_p,
    bankruptcy_threshold=10.0,
    min_open_dollars=2.0,
    transaction_eps=1e-2,
    use_dollar_volume=True,
    size_buckets=(0.50, 1.00),
    close_fee=1.0,
    open_fee=1.0,
    tax_rate=0.26,
    reward_mode="log",
    val_coeff=10.0,
    roi_coeff=100.0,
    done_reward_penalty=100.0,
    dtype=torch.float32,
    eps=1e-8,
)

env = LongShortEnv(**shared_kwargs, save_history=False)
state0 = env.reset(history[0])
mask0 = env.valid_action_mask()

print(f"state_dim: {env.state_dim} - action_dim: {env.action_dim}")
print(f"valid actions at reset: {int(mask0.sum().item())} / {env.action_dim}")
print("decode examples:")
for i in range(env.action_dim):
    print(f"action {i}: {env.decode_action(i)}")

In [ ]:
from src.agent.discrete.trpo import RecurrentTRPOAgent

agent = RecurrentTRPOAgent(
    state_dim=env.state_dim,
    action_dim=env.action_dim,
    hidden_dims=[512],
    hidden_dims_actor=[512, 512],
    hidden_dims_value=[512, 512],
    activation=torch.tanh,
    recurrent_type="simple",
    recurrent_kwargs={"memory_size": 64, "theta": 100_000},
    gamma=0.9999,
    vf_coef=0.5,
    ent_coef=0.05,
    advantage_type="gae",
    gae_lambda=0.95,
    normalize_advantages=False,
    dtype=env.dtype,
    device="cpu",
)

CHECKPOINT_PATH = f"../data/agent/trpo_{agent.net.recurrent_type}_longshort_buckets.ptm"
LOAD_CHECKPOINT = True
SAVE_CHECKPOINT = False

if LOAD_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
    agent.load(CHECKPOINT_PATH)
    print(f"loaded checkpoint: {CHECKPOINT_PATH}")
else:
    print("starting from scratch")

In [ ]:
train_cfg = dict(
    n_episodes=1,
    update_interval=100,
    n_updates=1,
    burn_in_updates=1,
    max_steps=50_000,
    warm_up=10_000,
    lr=1e-5,
    optim="AdamW",
    init_optimizer=False,
    store_results=True,
    max_grad_norm=10.0,
)

train_cfg

In [ ]:
loss, rewards, info = [], [], []

T_train = int(T_total * 0.9)

_loss, _rewards, _info = agent.train_on_historical(
    env,
    history[:T_train],
    **train_cfg,
)

if SAVE_CHECKPOINT:
    os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
    agent.save(CHECKPOINT_PATH)
    print(f"saved checkpoint: {CHECKPOINT_PATH}")

loss    += _loss
rewards += _rewards
info    += _info

len(loss), len(rewards), len(info)

In [ ]:
metric_names = next((list(ep_loss[0].keys()) for ep_loss in loss if len(ep_loss) > 0), [])
if not metric_names:
    raise ValueError("No loss metrics recorded yet.")

ref_metric = metric_names[0]
t_all = np.concatenate([
    np.linspace(i, i + 1, sum(len(np.asarray(loss_dict[ref_metric]).reshape(-1)) for loss_dict in ep_loss), endpoint=False)
    for i, ep_loss in enumerate(loss)
    if len(ep_loss) > 0
])

def flatten_metric(metric_name):
    chunks = []
    for ep_loss in loss:
        if len(ep_loss) == 0:
            continue
        chunks.append(np.concatenate([
            np.asarray(loss_dict[metric_name]).reshape(-1)
            for loss_dict in ep_loss
        ]))
    return np.concatenate(chunks) if chunks else np.array([])

palette = Category10[10]
figs = []
for i, metric_name in enumerate(metric_names):
    values = flatten_metric(metric_name)
    fig = bk.figure(
        title=metric_name.replace("_", " ").title(),
        x_axis_label="Training Iteration [Epochs]",
        y_axis_label="Value",
        width=1000,
        height=260,
    )
    fig.line(t_all[:len(values)], values, line_width=2, color=palette[i % len(palette)])
    figs.append(fig)

bk.show(bk.column(*figs))

In [ ]:
rewards_total = [sum(ep) for ep in rewards]
rewards_mean = [sum(ep) / max(1, len(ep)) for ep in rewards]

fig_total = bk.figure(
    title="Total Reward per Episode",
    x_axis_label="Training Iteration [Episodes]",
    y_axis_label="Total Reward",
    width=900,
    height=320,
)
fig_total.line(list(range(len(rewards_total))), rewards_total, line_width=2, color=Category10[10][4], legend_label="Total Reward / Episode")
fig_total.legend.location = "bottom_right"

fig_mean = bk.figure(
    title="Average Step Reward per Episode",
    x_axis_label="Training Iteration [Episodes]",
    y_axis_label="Reward",
    width=900,
    height=320,
)
fig_mean.line(list(range(len(rewards_mean))), rewards_mean, line_width=2, color=Category10[10][5], legend_label="Average Reward / Episode")
fig_mean.legend.location = "bottom_right"

bk.show(bk.column(fig_total, fig_mean))

In [ ]:
def build_rollout(infos, rewards_, asset_names_):
    if len(infos) == 0:
        raise ValueError("Rollout produced no steps.")

    ts = [datetime.fromtimestamp(item["t"]) for item in infos]
    Vs = torch.tensor([item["V"] for item in infos], dtype=torch.float32)
    Cs = torch.tensor([item["C"] for item in infos], dtype=torch.float32)
    ps = torch.tensor([item["p"] for item in infos], dtype=torch.float32)
    pos_units = torch.tensor([item["pos_units"] for item in infos], dtype=torch.float32)
    committed = torch.tensor([item["committed"] for item in infos], dtype=torch.float32)
    rewards_t = torch.tensor(rewards_, dtype=torch.float32)

    action_types = torch.tensor([item["action_type"] for item in infos], dtype=torch.long)
    action_assets = torch.tensor([item["action_asset"] for item in infos], dtype=torch.long)
    valid_trades = torch.tensor([float(item["valid_trade"]) for item in infos], dtype=torch.float32)
    realized_pnl = torch.tensor([sum(item["realized_pnl"]) for item in infos], dtype=torch.float32)
    realized_cost = torch.tensor([sum(item["realized_cost"]) for item in infos], dtype=torch.float32)

    portfolio_frac = pos_units * ps / Vs[:, None].clamp_min(1e-8)
    cash_frac = Cs / Vs.clamp_min(1e-8)
    cum_reward = rewards_t.cumsum(0)
    cum_realized_pnl = realized_pnl.cumsum(0)
    cum_realized_cost = realized_cost.cumsum(0)
    running_peak = torch.cummax(Vs, dim=0).values
    drawdown = 1.0 - Vs / running_peak.clamp_min(1e-8)
    norm_prices = ps / ps[0].clamp_min(1e-8)
    norm_value = Vs / Vs[0].clamp_min(1e-8)
    commitment_frac = committed.sum(dim=1) / (committed.sum(dim=1) + Cs).clamp_min(1e-8)
    gross_abs_exposure = portfolio_frac.abs().sum(dim=1)

    hold_mask = action_types == 0
    long_mask = action_types == 1
    short_mask = action_types == 2
    close_mask = action_types == 3
    trade_mask = long_mask | short_mask | close_mask
    invalid_mask = trade_mask & (valid_trades == 0)

    cumulative_longs = long_mask.to(torch.float32).cumsum(0)
    cumulative_shorts = short_mask.to(torch.float32).cumsum(0)
    cumulative_closes = close_mask.to(torch.float32).cumsum(0)
    cumulative_invalid = invalid_mask.to(torch.float32).cumsum(0)

    metrics = {
        "steps": len(infos),
        "final_value": float(Vs[-1].item()),
        "final_cash": float(Cs[-1].item()),
        "total_return_pct": float((norm_value[-1] - 1.0).item() * 100.0),
        "max_drawdown_pct": float(drawdown.max().item() * 100.0),
        "total_reward": float(cum_reward[-1].item()),
        "mean_reward": float(rewards_t.mean().item()),
        "reward_std": float(rewards_t.std(unbiased=False).item()),
        "long_count": int(long_mask.sum().item()),
        "short_count": int(short_mask.sum().item()),
        "close_count": int(close_mask.sum().item()),
        "hold_count": int(hold_mask.sum().item()),
        "invalid_trade_count": int(invalid_mask.sum().item()),
        "valid_trade_rate_pct": float(valid_trades.mean().item() * 100.0),
        "trade_rate_pct": float(trade_mask.to(torch.float32).mean().item() * 100.0),
        "realized_pnl_total": float(cum_realized_pnl[-1].item()),
        "realized_cost_total": float(cum_realized_cost[-1].item()),
        "final_commitment_pct": float(commitment_frac[-1].item() * 100.0),
        "max_abs_exposure_pct": float(gross_abs_exposure.max().item() * 100.0),
    }

    action_name_map = {0: "hold", 1: "long", 2: "short", 3: "close"}
    trade_rows = []
    for t, item in zip(ts, infos):
        action_type = int(item["action_type"])
        if action_type == 0:
            continue
        asset_idx = int(item["action_asset"])
        trade_rows.append({
            "t": t,
            "action": action_name_map[action_type],
            "asset": asset_names_[asset_idx] if 0 <= asset_idx < len(asset_names_) else None,
            "valid_trade": bool(item["valid_trade"]),
            "cash": float(item["C"]),
            "value": float(item["V"]),
            "realized_pnl": float(sum(item["realized_pnl"])),
        })

    return {
        "asset_names": asset_names_,
        "ts": ts,
        "Vs": Vs,
        "Cs": Cs,
        "ps": ps,
        "pos_units": pos_units,
        "committed": committed,
        "rewards": rewards_t,
        "action_types": action_types,
        "action_assets": action_assets,
        "valid_trades": valid_trades,
        "portfolio_frac": portfolio_frac,
        "cash_frac": cash_frac,
        "cum_reward": cum_reward,
        "cum_realized_pnl": cum_realized_pnl,
        "cum_realized_cost": cum_realized_cost,
        "drawdown": drawdown,
        "norm_prices": norm_prices,
        "norm_value": norm_value,
        "commitment_frac": commitment_frac,
        "gross_abs_exposure": gross_abs_exposure,
        "cumulative_longs": cumulative_longs,
        "cumulative_shorts": cumulative_shorts,
        "cumulative_closes": cumulative_closes,
        "cumulative_invalid": cumulative_invalid,
        "metrics": metrics,
        "trade_rows": trade_rows,
    }


def print_metrics(metrics):
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key:>22}: {value:.6f}")
        else:
            print(f"{key:>22}: {value}")


def show_rollout(rollout, title_prefix="Rollout"):
    ts = rollout["ts"]
    asset_names_ = rollout["asset_names"]
    norm_prices = rollout["norm_prices"]
    norm_value = rollout["norm_value"]
    Vs = rollout["Vs"]
    Cs = rollout["Cs"]
    committed = rollout["committed"]
    drawdown = rollout["drawdown"]
    rewards_t = rollout["rewards"]
    cum_reward = rollout["cum_reward"]
    cum_realized_pnl = rollout["cum_realized_pnl"]
    cum_realized_cost = rollout["cum_realized_cost"]
    cash_frac = rollout["cash_frac"]
    portfolio_frac = rollout["portfolio_frac"]
    commitment_frac = rollout["commitment_frac"]
    gross_abs_exposure = rollout["gross_abs_exposure"]
    cumulative_longs = rollout["cumulative_longs"]
    cumulative_shorts = rollout["cumulative_shorts"]
    cumulative_closes = rollout["cumulative_closes"]
    cumulative_invalid = rollout["cumulative_invalid"]

    overview_skip = max(1, len(ts) // 1500)
    reward_skip = max(1, len(ts) // 2000)
    reward_bins = min(200, max(20, len(rewards_t) // 10))
    alloc_skip = max(1, len(ts) // 1500)

    fig_prices = bk.figure(
        title=f"{title_prefix} - Normalized Prices vs Portfolio",
        width=1200,
        height=340,
        x_axis_type="datetime",
        x_axis_label="t",
        y_axis_label="Normalized Value [1]",
    )
    for i, name in enumerate(asset_names_):
        fig_prices.line(ts[::overview_skip], norm_prices[::overview_skip, i].numpy(), line_width=2, legend_label=name, color=Category10[10][i % 10])
    fig_prices.line(ts[::overview_skip], norm_value[::overview_skip].numpy(), line_width=3, legend_label="Portfolio", color="black")
    fig_prices.legend.click_policy = "hide"

    fig_value = bk.figure(
        title=f"{title_prefix} - Portfolio Value, Cash, Committed",
        width=1200,
        height=300,
        x_range=fig_prices.x_range,
        x_axis_type="datetime",
        x_axis_label="t",
        y_axis_label="EUR",
    )
    fig_value.line(ts, Vs.numpy(), line_width=2, legend_label="Portfolio Value", color=Category10[10][0])
    fig_value.line(ts, Cs.numpy(), line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][1])
    fig_value.line(ts, committed.sum(dim=1).numpy(), line_width=2, line_dash="dotted", legend_label="Committed", color=Category10[10][2])
    fig_value.legend.location = "bottom_right"

    fig_drawdown = bk.figure(
        title=f"{title_prefix} - Drawdown, Commitment, Abs Exposure",
        width=1200,
        height=280,
        x_range=fig_prices.x_range,
        x_axis_type="datetime",
        x_axis_label="t",
        y_axis_label="Percent [%]",
    )
    fig_drawdown.line(ts, (100.0 * drawdown).numpy(), line_width=2, color=Category10[10][3], legend_label="Drawdown")
    fig_drawdown.line(ts, (100.0 * commitment_frac).numpy(), line_width=2, line_dash="dashed", color=Category10[10][4], legend_label="Commitment")
    fig_drawdown.line(ts, (100.0 * gross_abs_exposure).numpy(), line_width=2, line_dash="dotted", color=Category10[10][5], legend_label="Abs Exposure")
    fig_drawdown.legend.location = "bottom_right"

    fig_reward = bk.figure(
        title=f"{title_prefix} - Step Reward",
        width=1200,
        height=280,
        x_axis_type="datetime",
        x_axis_label="t",
        y_axis_label="Reward",
    )
    fig_reward.line(ts[::reward_skip], rewards_t[::reward_skip].numpy(), line_width=2, color=Category10[10][4], legend_label="Step Reward")
    fig_reward.legend.location = "bottom_right"

    fig_cum_reward = bk.figure(
        title=f"{title_prefix} - Cumulative Reward",
        width=1200,
        height=280,
        x_range=fig_reward.x_range,
        x_axis_type="datetime",
        x_axis_label="t",
        y_axis_label="Cumulative Reward",
    )
    fig_cum_reward.line(ts, cum_reward.numpy(), line_width=2, color=Category10[10][2], legend_label="Cumulative Reward")
    fig_cum_reward.legend.location = "bottom_right"

    hist, edges = torch.histogram(rewards_t, bins=reward_bins, density=True)
    hist_x = ((edges[:-1] + edges[1:]) / 2).numpy()

    fig_hist = bk.figure(
        title=f"{title_prefix} - Reward Distribution",
        width=380,
        height=320,
        x_axis_label="Density",
        y_axis_label="Reward",
    )
    fig_hist.harea(y=hist_x, x1=0, x2=hist.numpy(), fill_color=Category10[10][4], fill_alpha=0.35)
    fig_hist.line(hist.numpy(), hist_x, line_color=Category10[10][4], line_width=2)

    fig_realized = bk.figure(
        title=f"{title_prefix} - Cumulative Realized PnL and Cost",
        width=800,
        height=320,
        x_range=fig_reward.x_range,
        x_axis_type="datetime",
        x_axis_label="t",
        y_axis_label="EUR",
    )
    fig_realized.line(ts, cum_realized_pnl.numpy(), line_width=2, color=Category10[10][0], legend_label="Cumulative Realized PnL")
    fig_realized.line(ts, cum_realized_cost.numpy(), line_width=2, color=Category10[10][1], line_dash="dashed", legend_label="Cumulative Realized Cost")
    fig_realized.legend.location = "bottom_right"

    fig_alloc = bk.figure(
        title=f"{title_prefix} - Signed Portfolio Fractions",
        width=1200,
        height=360,
        x_axis_type="datetime",
        x_axis_label="t",
        y_axis_label="Fraction [1]",
    )
    fig_alloc.line(ts[::alloc_skip], cash_frac[::alloc_skip].numpy(), line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
    for i, name in enumerate(asset_names_):
        fig_alloc.line(ts[::alloc_skip], portfolio_frac[::alloc_skip, i].numpy(), line_width=2, legend_label=name, color=Category10[10][(i + 1) % 10])
    fig_alloc.legend.click_policy = "hide"

    fig_actions = bk.figure(
        title=f"{title_prefix} - Cumulative Actions",
        width=1200,
        height=280,
        x_range=fig_alloc.x_range,
        x_axis_type="datetime",
        x_axis_label="t",
        y_axis_label="Count",
    )
    fig_actions.line(ts, cumulative_longs.numpy(), line_width=2, color=Category10[10][2], legend_label="Longs")
    fig_actions.line(ts, cumulative_shorts.numpy(), line_width=2, color=Category10[10][3], legend_label="Shorts")
    fig_actions.line(ts, cumulative_closes.numpy(), line_width=2, color=Category10[10][4], legend_label="Closes")
    fig_actions.line(ts, cumulative_invalid.numpy(), line_width=2, color=Category10[10][5], legend_label="Invalid")
    fig_actions.legend.location = "bottom_right"

    fig_committed = bk.figure(
        title=f"{title_prefix} - Committed Capital per Asset",
        width=1200,
        height=280,
        x_range=fig_alloc.x_range,
        x_axis_type="datetime",
        x_axis_label="t",
        y_axis_label="EUR",
    )
    for i, name in enumerate(asset_names_):
        fig_committed.line(ts[::alloc_skip], committed[::alloc_skip, i].numpy(), line_width=2, legend_label=name, color=Category10[10][i % 10])
    fig_committed.legend.click_policy = "hide"
    fig_committed.legend.location = "bottom_right"

    bk.show(
        bk.column(
            fig_prices,
            fig_value,
            fig_drawdown,
            fig_reward,
            fig_cum_reward,
            bk.row(fig_hist, fig_realized),
            fig_alloc,
            fig_actions,
            fig_committed,
        )
    )

In [ ]:
episode = -1
training_rollout = build_rollout(info[episode], rewards[episode], asset_names)
print_metrics(training_rollout["metrics"])

In [ ]:
show_rollout(training_rollout, title_prefix=f"Training Episode {episode}")

In [ ]:
training_rollout["trade_rows"][-5:]

In [ ]:
T_val_start = int(T_total * 0.9)
T_val_end = min(T_total - 1, T_val_start + 10_000)

agent.net.reset(1)
env.save_history = False

hist_r = []
hist_i = []

state = env.reset(history[T_val_start])
done = False
for idx in range(T_val_start + 1, T_val_end + 1):
    action = agent.act(state.to_tensor(), valid_mask=env.valid_action_mask(), explore=False, grad_enabled=False)
    next_state, reward, done, info_t = env.step(action, data=history[idx])

    hist_r.append(float(reward))
    hist_i.append(info_t)
    state = next_state

    if done:
        break

validation_rollout = build_rollout(hist_i, hist_r, asset_names)
print_metrics(validation_rollout["metrics"])
validation_rollout["metrics"]

In [ ]:
show_rollout(validation_rollout, title_prefix="Validation")

In [ ]:
validation_rollout["trade_rows"][-5:]